# 调用具体模型厂商api
## 1.1 deepseek

In [3]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)
from langchain_deepseek import ChatDeepSeek

llm_deepseek = ChatDeepSeek(
    model="deepseek-v4-flash",
)

response = llm_deepseek.invoke("你好")
# 用 Markdown 渲染，保留模型返回的换行和列表格式
display(Markdown(response.content))

你好！很高兴见到你😊 有什么我可以帮你的吗？无论是聊天、解答问题，还是需要一些建议，我都会尽力支持你！

## 1.2 千问

In [4]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)
from langchain_qwq import ChatQwen

llm_chatqwen = ChatQwen(
    model="qwen3.5-ocr",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

response = llm_chatqwen.invoke("你好")
# 打印 response 对象会挤成一行，应取 content 并渲染
display(Markdown(response.content))

```json
{
    "text": "你好"
}
```

## 1.3兼容的写法
使用ChatOpenAI

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

llm_openai = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPEN_302_API_KEY"),
    base_url=os.getenv("OPEN_302_API_BASE"),
)

response = llm_openai.invoke("你好")
display(Markdown(response.content))


你好！有什么我可以帮助你的吗？

## 1.4 模型初始化方案 init_chat_model()

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

llm = init_chat_model(
    model="gpt-4o-mini",
    api_key=os.getenv("OPEN_302_API_KEY"),
    base_url=os.getenv("OPEN_302_API_BASE"),
)

response = llm.invoke("你好")
display(Markdown(response.content))

你好！有什么我可以帮助你的吗？

## 1.5 本地模型

In [32]:
from langchain_ollama import ChatOllama

llm_ollama = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434"
)

response = llm_ollama.invoke("你好")
display(Markdown(response.content))


你好！有什么问题可以随时和我交流。我是阿里云开发的预训练模型，目前可以回答一些知识性、技巧性和创意性的问题，并能根据您的指令生成文字。例如文章、剧本、小说章节等。您需要了解些什么吗？或者我们来聊聊天？

# 模型使用
## 模型初始化参数

以下参数适用于 `ChatOpenAI`、`ChatDeepSeek`、`ChatQwen`、`init_chat_model()` 等 LangChain Chat Model。

### 一、连接与认证（几乎所有厂商通用）

| 参数 | 类型 | 说明 | 示例 |
|------|------|------|------|
| `model` | `str` | 模型名称（必填） | `"gpt-4o-mini"`、`"deepseek-v4-flash"` |
| `api_key` | `str` | API 密钥，未传时通常从环境变量读取 | `os.getenv("OPENAI_API_KEY")` |
| `base_url` | `str` | 自定义 API 地址（代理/兼容接口/国内站） | `"https://dashscope.aliyuncs.com/compatible-mode/v1"` |
| `timeout` | `float` | 请求超时时间（秒） | `30` |
| `max_retries` | `int` | 失败重试次数 | `2` |
| `default_headers` | `dict` | 自定义 HTTP 请求头 | `{"Authorization": "Bearer xxx"}` |

In [30]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)
from langchain_qwq import ChatQwen

llm_chatqwen = ChatQwen(
    model="deepseek-v4-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    temperature=1
)

response = llm_chatqwen.invoke("你是谁")
# 打印 response 对象会挤成一行，应取 content 并渲染
display(response.content)

'你好！我是DeepSeek，由深度求索公司创造的AI助手。😊\n\n我是一个纯文本模型，可以帮你解答各种问题、进行对话、处理文档等。我支持：\n- 📝 阅读链接和处理上传的文件（图片、PDF、Word、Excel、PPT等）\n- 🌐 联网搜索（需要手动开启）\n- 💬 上下文对话，最多支持1M token的超长文本\n- 🎤 App端的语音输入\n\n目前我是完全免费的，没有任何收费计划。有什么我可以帮你的吗？无论是学习、工作还是日常问题，尽管问我！✨'

# 模型的调用
- invoke(): 阻塞式，一次性返回完整结果回答
- ainvoke(): 非阻塞，提高系统吞吐量高并发web应用、I/O密集型任务
- stream(): 流式传输，实时返回每个token,聊天机器热、长文本生成
- astream(): 非阻塞，提高系统吞吐量高并发web应用、I/O密集型任务
- batch(): 批量处理，适合需要处理大量请求的场景
- abatch(): 非阻塞，提高系统吞吐量高并发web应用、I/O密集型任务

> `from langchain_core.messages import HumanMessage, AIMessage, SystemMessage`,有3个对象可以用来创建消息。`HumanMessage(content="xxx")`

## 2.1 invoke()
`response=model.invoke(input,config=one)`

- input: 输入文本
  - 字符串
  - 列表[str]
  - 字典[{"role":"user","content":"你好"},{"role":"assistant","content":"你好"}]
- config: 配置参数(dict)
  - max_tokens: 最大token数
  - temperature: 温度
  - top_p: 上文概率
  - top_k: 上文数量
- response: 返回结果

## 2.2 stream()
